# DEEPIFY – Google Colab Model Training Notebook
## Deep Learning-Based Deepfake Detection Using CNN & XceptionNet

This notebook trains the **DEEPIFY** core computer vision model on GPU hardware.

### Training Steps:
1. **GPU Acceleration Verification**
2. **Dataset Acquisition & Organization (FaceForensics++ / Celeb-DF)**
3. **Data Pipeline Construction with Xception Normalization `[-1, 1]`**
4. **Model Architecture: Transfer Learning via XceptionNet**
5. **Stage 1: Training Forensic Dense Head**
6. **Stage 2: Fine-Tuning Upper Convolutions (`block12` through `block14`)**
7. **Evaluation: ROC-AUC, Precision, Recall, F1, Confusion Matrix**
8. **Export `.keras` Model Binary & Forensic Telemetry**

### Step 1: Verify Google Colab GPU Environment
Ensure GPU is activated: **Runtime > Change runtime type > Hardware accelerator > T4 GPU**.

In [ ]:
!nvidia-smi

import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"[SUCCESS] GPU Detected: {gpus}")
else:
    print("[WARN] No GPU detected. Please navigate to Runtime > Change runtime type > T4 GPU.")

### Step 2: Mount Google Drive or Upload Dataset

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Optional: Mount Google Drive to load full FaceForensics++ or Celeb-DF archive
# drive.mount('/content/drive')

# Or unzip uploaded dataset.zip directly in Colab local workspace
if not os.path.exists('dataset'):
    print("Creating dataset directory structure or unzipping...")
    # If you uploaded a dataset.zip file, uncomment below:
    # !unzip -q /content/dataset.zip -d /content/
    !mkdir -p dataset/train/real dataset/train/fake dataset/validation/real dataset/validation/fake dataset/test/real dataset/test/fake

print("Dataset directories ready.")

### Step 3: Install Required Dependencies

In [ ]:
!pip install -q opencv-python-headless matplotlib seaborn scikit-learn pillow

### Step 4: High-Performance `tf.data` Pipeline with Normalization

In [ ]:
from tensorflow.keras import layers

IMG_SIZE = (299, 299)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset/train',
    labels='inferred',
    label_mode='binary',
    class_names=['real', 'fake'],
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset/validation',
    labels='inferred',
    label_mode='binary',
    class_names=['real', 'fake'],
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset/test',
    labels='inferred',
    label_mode='binary',
    class_names=['real', 'fake'],
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Conservative data augmentations that protect manipulation seams
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.04)
])

# Standard XceptionNet input normalization: [-1.0, 1.0]
norm_layer = layers.Rescaling(scale=1.0 / 127.5, offset=-1.0)

train_ds = train_ds.map(lambda x, y: (norm_layer(data_augmentation(x, training=True)), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (norm_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(lambda x, y: (norm_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

print("Data pipelines configured successfully.")

### Step 5: Build Deepify XceptionNet Architecture

In [ ]:
from tensorflow.keras.applications import Xception
from tensorflow.keras import models, regularizers

inputs = layers.Input(shape=(299, 299, 3), name="input_face")

# Pretrained feature extractor
base_model = Xception(weights='imagenet', include_top=False, input_tensor=inputs)
base_model.trainable = False

# Forensic Classification Head
x = layers.GlobalAveragePooling2D(name="gap")(base_model.output)
x = layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="dense_512")(x)
x = layers.BatchNormalization(name="bn_1")(x)
x = layers.Dropout(0.5, name="drop_1")(x)
x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="dense_128")(x)
x = layers.Dropout(0.3, name="drop_2")(x)
outputs = layers.Dense(1, activation="sigmoid", name="prediction")(x)

model = models.Model(inputs=inputs, outputs=outputs, name="Deepify_XceptionNet")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.AUC(name="auc")]
)

model.summary()

### Step 6: Stage 1 Training – Transfer Learning with Callbacks

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

os.makedirs('saved_models', exist_ok=True)
checkpoint_path = 'saved_models/xception_deepfake.keras'

callbacks_list = [
    ModelCheckpoint(checkpoint_path, monitor='val_loss', mode='min', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1)
]

print("Starting Stage 1: Feature Extraction Head Training...")
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=callbacks_list
)

### Step 7: Stage 2 Training – Fine-Tuning Upper Convolutions

In [ ]:
# Unfreeze layers from block12 onwards
trainable_flag = False
for layer in model.layers:
    if 'block12' in layer.name:
        trainable_flag = True
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    elif trainable_flag:
        layer.trainable = True

# Recompile with lower learning rate for delicate boundary fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.AUC(name="auc")]
)

print("Starting Stage 2: Fine-Tuning upper blocks...")
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=6,
    callbacks=callbacks_list
)

### Step 8: Comprehensive Test Evaluation & Metrics Generation

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Load best checkpointed model
best_model = tf.keras.models.load_model(checkpoint_path)

y_true = []
y_probs = []
for batch_x, batch_y in test_ds:
    preds = best_model.predict(batch_x, verbose=0)
    y_true.extend(batch_y.numpy().flatten())
    y_probs.extend(preds.flatten())

y_true = np.array(y_true)
y_probs = np.array(y_probs)
y_pred = (y_probs >= 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['REAL', 'FAKE']))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['REAL', 'FAKE'], yticklabels=['REAL', 'FAKE'])
plt.title('DEEPIFY – Test Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig('saved_models/confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

auc = roc_auc_score(y_true, y_probs)
print(f"ROC-AUC Score: {auc:.4f}")

### Step 9: Download Trained Model & Artifacts

In [ ]:
from google.colab import files

# Download the final trained model for local VS Code inference
files.download('saved_models/xception_deepfake.keras')
files.download('saved_models/confusion_matrix.png')